# SDTM_Variables — the SDTM v2.0 Model and SDTMIG v3.4 variable tables, as published

Builds `sdtm-domain-reference/machine_actionable/SDTM_Variables.xlsx`: the variables of the general
observation classes (Interventions, Events, Findings, Findings About) as the SDTM v2.0 Model and the
SDTMIG v3.4 variable tables publish them, joined so that each domain variable carries what the Model
says about its generic variable — role, the variables it qualifies, definition, usage restriction —
and the NCIt concept of the Model variable (C-code, preferred term, parents).

Facts only. Nothing is classified, interpreted or decided here. The Model's `Variables Qualified`
says what a Variable Qualifier qualifies (e.g. --TSTDTL qualifies --TESTCD, --LAT qualifies --SPEC
and --LOC); a Record Qualifier qualifies the record as a whole and names no variable.

## Inputs

| File | Used for |
|---|---|
| `sdtm-domain-reference/downloads/SDTM_v2.0.csv` | the Model variable table (gitignored; downloaded with a cdisc.org sign-in) |
| `sdtm-domain-reference/downloads/SDTMIG_v3.4.csv` | the IG variable tables (gitignored; same) |
| `sdtm-test-codes/downloads/Thesaurus.txt`, `Thesaurus.FLAT.zip` | NCIt preferred term and parents of the Model variable C-codes |
| `sdtm-domain-reference/machine_actionable/SDTMIG_Statements.xlsx` | the IG statements that name a variable |
| `Reference_Versions.json` | SDTMIG, SDTM and NCIt pins |

## Output

`sdtm-domain-reference/machine_actionable/SDTM_Variables.xlsx` — sheets `ReadMe`, `Model_Variables`,
`Domain_Variables`, `Provenance`.

The run **fails** when: a pin is not the version this notebook reads; a CSV is another version; the
FLAT file is not the pinned build; a Model C-code is not in the FLAT file; a class and variable occur
twice in the Model, or a domain and variable twice in the IG; SDTMIG_Statements was built on another
SDTMIG version.

In [1]:
import hashlib
import json
import zipfile
from datetime import date, datetime
from pathlib import Path

import pandas as pd
from openpyxl import Workbook, load_workbook
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side

BASE_DIR = Path.cwd().parent          # sdtm-domain-reference/
REPO_ROOT = BASE_DIR.parent           # cdisc-for-ai/
VERSIONS = json.loads((REPO_ROOT / 'Reference_Versions.json').read_text(encoding='utf-8'))
MODEL_CSV = BASE_DIR / 'downloads' / 'SDTM_v2.0.csv'
IG_CSV = BASE_DIR / 'downloads' / 'SDTMIG_v3.4.csv'
FLAT_ZIP = REPO_ROOT / 'sdtm-test-codes' / 'downloads' / 'Thesaurus.FLAT.zip'
FLAT_FILE = REPO_ROOT / 'sdtm-test-codes' / 'downloads' / 'Thesaurus.txt'
IG_STATEMENTS_FILE = BASE_DIR / 'machine_actionable' / 'SDTMIG_Statements.xlsx'
OUTPUT_FILE = BASE_DIR / 'machine_actionable' / 'SDTM_Variables.xlsx'

# The general observation classes: their own variables, and the variables they share.
OBSERVATION_CLASSES = ['Interventions', 'Events', 'Findings', 'Findings About']
SHARED_CLASS = 'General Observations'

for path in [MODEL_CSV, IG_CSV, FLAT_ZIP, FLAT_FILE, IG_STATEMENTS_FILE]:
    if not path.exists():
        raise FileNotFoundError(path)
if VERSIONS['SDTMIG'] != '3.4':
    raise RuntimeError(f'This notebook reads SDTMIG v3.4; the pin is {VERSIONS["SDTMIG"]}')
if VERSIONS['SDTM'] != '2.0':
    raise RuntimeError(f'This notebook reads SDTM v2.0; the pin is {VERSIONS["SDTM"]}')

model = pd.read_csv(MODEL_CSV, dtype=str, keep_default_na=False)
if set(model['Version']) != {'SDTM v2.0'}:
    raise RuntimeError(f'{MODEL_CSV.name}: Version {sorted(set(model["Version"]))}')
ig = pd.read_csv(IG_CSV, dtype=str, keep_default_na=False)
if set(ig['Version']) != {'SDTMIG v3.4'}:
    raise RuntimeError(f'{IG_CSV.name}: Version {sorted(set(ig["Version"]))}')

with zipfile.ZipFile(FLAT_ZIP) as zf:
    _info = zf.getinfo('Thesaurus.txt')
FLAT_DATE = date(*_info.date_time[:3]).isoformat()
if _info.file_size != FLAT_FILE.stat().st_size:
    raise RuntimeError(f'{FLAT_FILE.name} is not the file in {FLAT_ZIP.name}: unzip again')
if FLAT_DATE != VERSIONS['NCIt']:
    raise RuntimeError(f"NCIt pin {VERSIONS['NCIt']} != FLAT build date {FLAT_DATE}")

_ig_prov = pd.read_excel(IG_STATEMENTS_FILE, sheet_name='Provenance', dtype=str, keep_default_na=False)
if dict(zip(_ig_prov['Key'], _ig_prov['Value']))['SDTMIG'] != VERSIONS['SDTMIG']:
    raise RuntimeError(f'{IG_STATEMENTS_FILE.name} was built on another SDTMIG version: re-run SDTMIG_Statements')

print(f'{MODEL_CSV.name}: {len(model):,} rows · {IG_CSV.name}: {len(ig):,} rows · NCIt FLAT {FLAT_DATE} (= pin)')

SDTM_v2.0.csv: 527 rows · SDTMIG_v3.4.csv: 1,917 rows · NCIt FLAT 2026-09-29 (= pin)


## 1. Model_Variables

Every SDTM v2.0 variable of the general observation classes and of General Observations (the
variables the classes share), as published, plus the NCIt concept of its `Variable C-Code`:
preferred term (first FLAT synonym) and parents (codes and preferred terms, `|`-separated).
Variables without a C-code get no NCIt columns.

In [2]:
model_variables = model[model['Class'].isin(OBSERVATION_CLASSES + [SHARED_CLASS])].copy()
dup = model_variables[model_variables.duplicated(['Class', 'Variable Name'], keep=False)]
if len(dup):
    raise RuntimeError(f'{MODEL_CSV.name}: a class and variable occur more than once:\n{dup[["Class", "Variable Name"]]}')

FLAT_COLS = ['code', 'url', 'parents', 'synonyms', 'definition',
             'display_name', 'concept_status', 'semantic_type', 'subset_membership']
flat = pd.read_csv(FLAT_FILE, sep='\t', header=None, names=FLAT_COLS, dtype=str, na_filter=False)
pref_name = {c: v.split('|')[0].strip() for c, v in zip(flat['code'], flat['synonyms'])}
parents_of = dict(zip(flat['code'], flat['parents']))

ccodes = sorted(set(model_variables['Variable C-Code']) - {''})
missing = [c for c in ccodes if c not in pref_name]
if missing:
    raise RuntimeError(f'Model variable C-codes not in the FLAT file: {missing}')


def ncit(code):
    if code == '':
        return pd.Series({'NCIt_Preferred_Term': '', 'NCIt_Parent_Codes': '', 'NCIt_Parent_Terms': ''})
    parents = [p for p in parents_of[code].split('|') if p]
    no_name = [p for p in parents if not pref_name.get(p)]
    if no_name:
        raise RuntimeError(f'{code}: parent codes without a FLAT row: {no_name}')
    return pd.Series({'NCIt_Preferred_Term': pref_name[code], 'NCIt_Parent_Codes': '|'.join(parents),
                      'NCIt_Parent_Terms': '|'.join(pref_name[p] for p in parents)})


model_variables = pd.concat([model_variables.reset_index(drop=True),
                             model_variables['Variable C-Code'].reset_index(drop=True).apply(ncit)], axis=1)
print(f'Model_Variables: {len(model_variables)} rows; with a C-code: {(model_variables["Variable C-Code"] != "").sum()}')
print(model_variables.groupby('Class').agg(Variables=('Variable Name', 'size'),
                                           With_CCode=('Variable C-Code', lambda s: (s != '').sum())).to_string())

Model_Variables: 264 rows; with a C-code: 164
                      Variables  With_CCode
Class                                      
Events                       56          41
Findings                    100          47
Findings About                1           0
General Observations         64          48
Interventions                43          28


## 2. Domain_Variables

Every SDTMIG v3.4 variable of a domain in the general observation classes, as published, with the
Model facts of its generic variable beside it.

**Model lookup.** The generic variable is the IG name itself when the Model has it (STUDYID, VISITNUM,
EPOCH), else `--` + the name without the domain code (LBSPEC → `--SPEC`). It is looked up in the
domain's IG class, then — for Findings About — in Findings, then in General Observations. The Findings
step for Findings About is the lookup this file uses, not a statement of the Model; the Model lists
only `--OBJ` under Findings About. Not found: `Model_Role` = `(not in SDTM v2.0)`.

`IG_Role_Differs_From_Model` = `yes` when both roles are given and differ (e.g. MITSTDTL).

**Statements.** `Statements_Naming`: the Statement_IDs of `SDTMIG_Statements.xlsx` that name the
variable or quote its CDISC Notes (its `Named_In`). A statement that names only the generic form of a
variable the domain does not have (EG-A11: `--SPEC`) is in `Model_Variables_Named` of SDTMIG_Statements.

In [3]:
domain_variables = ig[ig['Class'].isin(OBSERVATION_CLASSES)].copy()
dup = domain_variables[domain_variables.duplicated(['Dataset Name', 'Variable Name'], keep=False)]
if len(dup):
    raise RuntimeError(f'{IG_CSV.name}: a domain and variable occur more than once:\n{dup[["Dataset Name", "Variable Name"]]}')

model_index = model_variables.set_index(['Class', 'Variable Name'])
model_names = set(model_variables['Variable Name'])


def generic(dataset, variable):
    if variable in model_names:
        return variable
    if variable.startswith(dataset):
        return '--' + variable[len(dataset):]
    return variable


def lookup_classes(ig_class):
    return [ig_class] + (['Findings'] if ig_class == 'Findings About' else []) + [SHARED_CLASS]


MODEL_FIELDS = {'Class': 'Model_Class', 'Role': 'Model_Role', 'Variables Qualified': 'Model_Variables_Qualified',
                'Usage Restrictions': 'Model_Usage_Restrictions', 'Variable C-Code': 'Model_Variable_CCode',
                'NCIt_Preferred_Term': 'NCIt_Preferred_Term', 'NCIt_Parent_Codes': 'NCIt_Parent_Codes',
                'NCIt_Parent_Terms': 'NCIt_Parent_Terms', 'Definition': 'Model_Definition', 'Notes': 'Model_Notes'}


def model_facts(r):
    g = generic(r['Dataset Name'], r['Variable Name'])
    out = {'Generic_Variable': g}
    for c in lookup_classes(r['Class']):
        if (c, g) in model_index.index:
            m = model_index.loc[(c, g)]
            out.update({new: (c if old == 'Class' else m[old]) for old, new in MODEL_FIELDS.items()})
            return pd.Series(out)
    out.update({new: '' for new in MODEL_FIELDS.values()})
    out['Model_Role'] = '(not in SDTM v2.0)'
    return pd.Series(out)


domain_variables = pd.concat([domain_variables.reset_index(drop=True),
                              domain_variables.reset_index(drop=True).apply(model_facts, axis=1)], axis=1)
in_model = domain_variables['Model_Role'] != '(not in SDTM v2.0)'
domain_variables['IG_Role_Differs_From_Model'] = [
    'yes' if m and r != mr else '' for m, r, mr in zip(in_model, domain_variables['Role'], domain_variables['Model_Role'])]

st_vars = pd.read_excel(IG_STATEMENTS_FILE, sheet_name='Variables', dtype=str, keep_default_na=False)
named_in = dict(zip(zip(st_vars['Dataset Name'], st_vars['Variable Name']), st_vars['Named_In']))
domain_variables['Statements_Naming'] = [named_in.get((d, v), '') for d, v in
                                         zip(domain_variables['Dataset Name'], domain_variables['Variable Name'])]

print(f'Domain_Variables: {len(domain_variables):,} rows in {domain_variables["Dataset Name"].nunique()} domains')
print(f'  not in SDTM v2.0: {(~in_model).sum()}; IG role differs from the Model: '
      f'{", ".join(domain_variables.loc[domain_variables["IG_Role_Differs_From_Model"] == "yes", "Variable Name"]) or "(none)"}')
print(f'  named by a statement: {(domain_variables["Statements_Naming"] != "").sum()}')
print(domain_variables.groupby(['Class', 'Role']).size().unstack(fill_value=0).to_string())
print('\nNot in SDTM v2.0:', ', '.join(domain_variables.loc[~in_model, 'Variable Name']))

Domain_Variables: 1,739 rows in 46 domains
  not in SDTM v2.0: 0; IG role differs from the Model: AGROUTE, CMROUTE, ECROUTE, EXROUTE, PRROUTE, SUDOSFRQ, SUROUTE, BEPRTYID, CPBDAGNT, CPCOLSRT, CPLOINC, LBCOLSRT, LBLOINC, MBLOINC, MITSTDTL, MSAGENT, MSLOINC, VSLAT
  named by a statement: 69
Role            Grouping Qualifier  Identifier  Record Qualifier  Result Qualifier  Synonym Qualifier  Timing  Topic  Variable Qualifier
Class                                                                                                                                   
Events                          14          50                42                 0                 10      66      7                  15
Findings                        55         236               256                86                 35     323     30                 162
Findings About                   4          13                17                 6                  2      19      2                   6
Interventions            

## 3. Write the workbook

In [4]:
HEADER_FONT = Font(name='Arial', bold=True, size=10, color='FFFFFF')
DATA_FONT = Font(name='Arial', size=10)
GREY = PatternFill('solid', fgColor='7F7F7F')
PALE = PatternFill('solid', fgColor='F2F2F2')
BAND = PatternFill('solid', fgColor='595959')
THIN = Side(style='thin', color='999999')
BORDER = Border(left=THIN, right=THIN, top=THIN, bottom=THIN)
WRAP = Alignment(wrap_text=True, vertical='top')
WIDE = {'Definition': 80, 'Notes': 80, 'CDISC Notes': 80, 'Model_Definition': 60, 'Model_Notes': 60, 'Examples': 50}

n_not_in_model = int((~in_model).sum())
readme_lines = [
    'SDTM_Variables — the SDTM v2.0 Model and SDTMIG v3.4 variable tables, as published',
    '',
    'PURPOSE',
    'The variables of the general observation classes as the SDTM v2.0 Model and the SDTMIG v3.4 variable tables',
    '  publish them, joined: each domain variable with what the Model says about its generic variable -- role,',
    '  the variables it qualifies, definition, usage restriction -- and the NCIt concept of the Model variable.',
    '  So a consumer reads one file for what qualifies the test (--TESTCD), the specimen or location, the result',
    '  or another variable, and what qualifies only the record, instead of the CDISC CSVs.',
    '  Facts only: nothing is classified, interpreted or decided here.',
    '',
    'SCOPE',
    f'Model_Variables: {len(model_variables)} SDTM v2.0 variables of {", ".join(OBSERVATION_CLASSES)} and {SHARED_CLASS}.',
    f'Domain_Variables: {len(domain_variables):,} SDTMIG v3.4 variables in {domain_variables["Dataset Name"].nunique()} domains of those classes,',
    '  every role. Special-Purpose, Trial Design, Relationship and Study Reference datasets are not here.',
    '',
    'SOURCE VERSIONS',
    f'SDTM v{VERSIONS["SDTM"]} · SDTMIG v{VERSIONS["SDTMIG"]} (2021-11-29) · NCIt FLAT {VERSIONS["NCIt"]} · pinned in Reference_Versions.json (repo root).',
    '  As keys for programs: sheet Provenance.',
    f'Generated: {datetime.now():%Y-%m-%d %H:%M}',
    'Notebook: sdtm-domain-reference/notebooks/SDTM_Variables.ipynb',
    'Inputs: SDTM_v2.0.csv and SDTMIG_v3.4.csv (CDISC, downloaded with a cdisc.org sign-in; not in the repo);',
    '  the NCIt FLAT file (NCI EVS); sdtm-domain-reference/machine_actionable/SDTMIG_Statements.xlsx.',
    'Variable-table content © Clinical Data Interchange Standards Consortium, Inc. Exploratory work, not a CDISC product.',
    '',
    'KNOWN GAPS',
    'The Model gives a C-code for some variables only (e.g. --POS, --SPEC, --LEAD, --FAST; not --METHOD, --LOC,',
    '  --TSTDTL). NCIt places a C-code under NCI\'s parents; that is NCI\'s classification, not CDISC\'s.',
    'The roles do not separate what identifies a test from what qualifies the record: --SPEC, --METHOD, --POS and',
    '  --FAST are all Record Qualifiers. Where a domain says more (LB Assumption 8), see SDTMIG_Statements.',
    'Findings About domains are looked up in Findings after Findings About; the Model lists only --OBJ under',
    '  Findings About. This lookup is the file\'s, not the Model\'s.',
    f'{n_not_in_model} IG variables have no generic variable in the Model (Model_Role "(not in SDTM v2.0)").',
    'Codelist Submission Values is empty on every row of SDTMIG_v3.4.csv as published, so it is empty here.',
    'CDISC Notes, Definition and Notes are as published, including the CSV\'s literal \\n paragraph markers.',
    '',
    'SHEETS',
    'Model_Variables — one row per SDTM v2.0 class and variable, as published, plus its NCIt concept.',
    'Domain_Variables — one row per SDTMIG v3.4 domain and variable, as published, plus the Model facts and the',
    '  statements that name it.',
    'Provenance — versions; per sheet rows, columns and a content hash.',
    '',
    'COLUMNS — Model_Variables',
    'The columns of SDTM_v2.0.csv, unchanged, plus:',
    'NCIt_Preferred_Term: the NCIt preferred term of Variable C-Code (first FLAT synonym).',
    'NCIt_Parent_Codes, NCIt_Parent_Terms: its NCIt parents, |-separated, in the same order.',
    '',
    'COLUMNS — Domain_Variables',
    'The columns of SDTMIG_v3.4.csv, unchanged, plus:',
    'Generic_Variable: the Model variable it was looked up as (--SPEC for LBSPEC; VISITNUM as is).',
    'Model_Class: the Model class it was found in (own class, Findings for Findings About, or General Observations).',
    'Model_Role, Model_Variables_Qualified, Model_Usage_Restrictions, Model_Variable_CCode, Model_Definition,',
    '  Model_Notes: the Model row\'s Role, Variables Qualified, Usage Restrictions, Variable C-Code, Definition, Notes.',
    'NCIt_Preferred_Term, NCIt_Parent_Codes, NCIt_Parent_Terms: as in Model_Variables.',
    'IG_Role_Differs_From_Model: yes when both roles are given and differ.',
    'Statements_Naming: Statement_IDs in SDTMIG_Statements that name the variable or quote its CDISC Notes.',
    '  A statement that names only a generic form the domain does not have (EG-A11: --SPEC, --FAST, --LOINC,',
    '  as qualifiers not generally used) is in Model_Variables_Named of SDTMIG_Statements.',
    '',
    'READING',
    'Read with keep_default_na=False (pandas) or the equivalent, so that no cell is read as missing.',
]

wb = Workbook()
ws = wb.active
ws.title = 'ReadMe'
for i, text in enumerate(readme_lines, 1):
    c = ws.cell(row=i, column=1, value=text)
    c.font = HEADER_FONT if i == 1 else Font(name='Arial', size=10, bold=text.isupper() and bool(text))
    if i == 1:
        c.fill = BAND
ws.column_dimensions['A'].width = 80


def write_sheet(name, df):
    ws = wb.create_sheet(name)
    for ci, col in enumerate(df.columns, 1):
        c = ws.cell(row=1, column=ci, value=col)
        c.font, c.fill, c.border, c.alignment = HEADER_FONT, GREY, BORDER, WRAP
    for ri, row in enumerate(df.itertuples(index=False, name=None), 2):
        for ci, v in enumerate(row, 1):
            c = ws.cell(row=ri, column=ci, value=v)
            c.font, c.fill, c.border, c.alignment = DATA_FONT, PALE, BORDER, WRAP
    for ci, col in enumerate(df.columns, 1):
        width = max([len(col)] + [len(str(v)) for v in df[col]]) + 2
        ws.column_dimensions[ws.cell(row=1, column=ci).column_letter].width = min(width, WIDE.get(col, 40))
    ws.freeze_panes = 'A2'
    ws.auto_filter.ref = ws.dimensions


write_sheet('Model_Variables', model_variables)
write_sheet('Domain_Variables', domain_variables)
OUTPUT_FILE.parent.mkdir(exist_ok=True)
wb.save(OUTPUT_FILE)
print(f'Written: {OUTPUT_FILE.relative_to(REPO_ROOT)}')

Written: sdtm-domain-reference/machine_actionable/SDTM_Variables.xlsx


In [5]:
# ── Provenance sheet ── (as in SDTMIG_Statements)
PROVENANCE_VERSION_KEYS = ['SDTM_CT', 'COSMoS', 'SDTMIG', 'SDTM', 'NCIt']
PROVENANCE_METHOD = ('SHA-256 (UTF-8) of the sheet read as text, empty cells as "" (pandas read_excel, dtype=str): '
                     'header and rows, cells joined by tab, rows by newline. Equal hash = equal data.')


def sheet_sha256(df):
    lines = ['\t'.join(df.columns)] + ['\t'.join(row) for row in df.itertuples(index=False, name=None)]
    return hashlib.sha256('\n'.join(lines).encode('utf-8')).hexdigest()


def add_provenance(path, notebook):
    """Add a Provenance sheet (Key, Value) to a saved workbook."""
    sheets = pd.read_excel(path, sheet_name=None, dtype=str, keep_default_na=False)
    if 'Provenance' in sheets:
        raise RuntimeError(f'{path.name} already has a Provenance sheet')
    rows = [(k, VERSIONS[k]) for k in PROVENANCE_VERSION_KEYS]
    rows += [('Generated', f'{datetime.now():%Y-%m-%d %H:%M}'), ('Notebook', notebook)]
    for name, df in sheets.items():
        if name == 'ReadMe':
            continue
        rows += [(f'{name}.Rows', str(len(df))), (f'{name}.Columns', '|'.join(df.columns)),
                 (f'{name}.Content_SHA256', sheet_sha256(df))]
    rows.append(('Content_SHA256.Method', PROVENANCE_METHOD))
    wb_p = load_workbook(path)
    ws_p = wb_p.create_sheet('Provenance')
    for ci, head in enumerate(['Key', 'Value'], 1):
        cell = ws_p.cell(row=1, column=ci, value=head)
        cell.font = HEADER_FONT
        cell.fill = GREY
    for ri, (k, v) in enumerate(rows, 2):
        ws_p.cell(row=ri, column=1, value=k).font = DATA_FONT
        ws_p.cell(row=ri, column=2, value=v).font = DATA_FONT
    ws_p.column_dimensions['A'].width = 34
    ws_p.column_dimensions['B'].width = 90
    ws_p.freeze_panes = 'A2'
    wb_p.save(path)
    print(f'Provenance: {path.name}, {len(rows)} keys')


add_provenance(OUTPUT_FILE, 'sdtm-domain-reference/notebooks/SDTM_Variables.ipynb')

Provenance: SDTM_Variables.xlsx, 14 keys


In [6]:
# Read back: the written sheets equal the frames.
back = pd.read_excel(OUTPUT_FILE, sheet_name=None, dtype=str, keep_default_na=False)
for name, df in [('Model_Variables', model_variables), ('Domain_Variables', domain_variables)]:
    if not back[name].equals(df.astype(str)):
        raise RuntimeError(f'{name}: read-back differs from what was written')
print('Read-back: Model_Variables and Domain_Variables as written.')

Read-back: Model_Variables and Domain_Variables as written.
